# Foundry IQ: Agentic Retrieval with a Topic Graph

## Summary

**You send a "theme question" that spans many documents to a Foundry IQ knowledge base, and you check in actual `retrieve` responses how a Topic Graph changes the way evidence is selected.**

A question such as "What are the main themes across the whole document set, covering history, buildings, annual events and tourism?" cannot be answered from a single paragraph. A search that returns only the chunks closest to the question can skew toward similar content. A Topic Graph selects evidence from multiple themes, which makes it easier to get evidence that reflects a broad range of themes.

| Component | Role | How this notebook uses it |
|---|---|---|
| Foundry IQ (Azure AI Search) | A service that lets agents and apps retrieve permission-aware, grounded information from scattered documents. You define what to search as a "knowledge source" and the entry point for search as a "knowledge base". | Creates one dedicated knowledge source and one dedicated knowledge base, and calls `retrieve`. |
| Agentic Retrieval | A multi-query pipeline for complex questions that runs when you request `retrieve` on a knowledge base. As needed, an LLM decomposes the question, searches in parallel, reranks, and merges the results into one. | To compare only the diversity setting under identical conditions, the notebook calls it with the reasoning effort fixed to `minimal`. It does not use LLM question decomposition, reranking or answer generation. |
| Topic Graph | A graph that represents groups of related chunks (communities) and their topic hierarchy. It samples evidence from multiple topics. | Connects to the chunk vector field and compares evidence selection with `none`, `balanced` and `diverse`. |

### Steps

1. **Chapter 1:** Check the connection settings and the switch that allows changes to Azure.
2. **2.1-2.2:** Check the registered chunk index and topic index, and connect the Topic Graph to the chunk vector field. The connection is added only when you review the plan and approve it.
3. **2.3:** Create the knowledge source and the knowledge base.
4. **2.4-2.8:** Search with the same question in three ways, and compare the returned chunks, source references and activity log side by side.

### Prerequisites and notes

- **Prerequisites:** The chunk index built in [Functional Test 2](./functional-test-2-topic-graph.ipynb) (default name `ft-topic-src`) and the topic index registered in Search in chapter 9 of the same Functional Test 2. This notebook does not build or re-register the graph. To try your own data, prepare the topic index that corresponds to that chunk index with the Foundry IQ skills plugin.
- **Preview:** Topic Graph is a public preview feature, and the target API is `2026-11-01-preview`.
- **How to read the results:** With few documents, the three methods can return the same results. This comparison checks settings and returned data. It is not an evaluation that guarantees search quality or improved diversity.

Source: [What is Foundry IQ?](https://learn.microsoft.com/azure/foundry/agents/concepts/what-is-foundry-iq) / [Agentic retrieval in Foundry IQ](https://learn.microsoft.com/azure/search/agentic-retrieval-overview) / [Topic Graph retrieval configuration](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph) / [Knowledge graph features overview](https://learn.microsoft.com/azure/search/knowledge-graph-features-overview)

## Background: Foundry IQ, Agentic Retrieval and Topic Graph

### What Foundry IQ provides

The model that powers an agent has a knowledge limit set by its training data cutoff, and it cannot access documents inside your organization on its own. Foundry IQ combines documents from multiple data stores into a single "knowledge base" and returns permission-aware, grounded information to agents and apps. Multiple agents can share one knowledge base.

Think of a library: **a knowledge source is the "stacks" (the book storage) and a knowledge base is the "librarian's desk".** Users only put a question to the desk, and the desk decides which stacks to search and how. You can point the stacks at Azure Blob Storage, SharePoint, OneLake, the public web and more. The only stacks in this notebook are one chunk index on the Search service.

```mermaid
flowchart LR
    U["Agent / app"] -->|"retrieve (question)"| KB["Knowledge base<br/>Librarian's desk"]
    KB --> KS1["Knowledge source<br/>This notebook: chunk index"]
    KB -.-> KS2["Other knowledge sources<br/>Blob, SharePoint, OneLake, Web"]
    KS1 --> R["Response<br/>Evidence excerpts, source references, activity log"]
    KS2 -.-> R
    R --> U
```

### What Agentic Retrieval does

Agentic Retrieval is a multi-query pipeline designed for complex questions from chats and agents. For example, a question that contains several requests, such as "a hotel near the beach, with an airport shuttle, within walking distance of a vegetarian restaurant", is hard to handle in a single search, so the question is split and searched in parallel. The response consists of evidence excerpts (`response`), source references (`references`) and the activity log (`activity`).

| Stage | Normal behavior | Setting in this notebook |
|---|---|---|
| 1. Planning | With `low` and `medium`, an LLM decomposes the question into subqueries and chooses the knowledge sources to search. | `minimal`: no LLM is used, and the question is passed as is, as a single search intent. |
| 2. Execution | Runs the subqueries in parallel and reranks each result with the semantic ranker. | Knowledge source `resultsProcessing = none`: skips reranking and keeps the order of the index results. |
| 3. Merging | Returns excerpts, or an LLM synthesizes an answer. | `extractiveData`: returns excerpts and source references and does not generate an answer. |

**The reasoning effort is fixed to `minimal` so that the diversity settings are compared under identical conditions.** Sections 2.5-2.7 use the same question and the same knowledge source and change only the `topicGraph` setting (`none`, `balanced`, `diverse`). With `low` and `medium`, an LLM decomposes the question into subqueries, so the searches that actually run can differ from request to request, and a difference in results can no longer be read as a difference in the diversity setting. `minimal` skips LLM search planning and passes the question as is, as a single search intent. `resultsProcessing = none`, which skips reranking, serves the same purpose: reranking does not affect the returned order.

This notebook uses the Agentic Retrieval framework (knowledge base, knowledge source, `retrieve`) while detaching the parts that use an LLM, so that you can observe only how a Topic Graph changes the way evidence is selected. An example that uses LLM search planning is covered in the [Entity Graph notebook](./agentic-retrieval-entity-graph.ipynb).

### What a Topic Graph solves

Vector search returns the chunks closest to the question, starting from the top. For a question about the whole set, such as "What themes does this document set contain?", if chunks with similar content take the top ranks, the other themes drop out of the evidence.

A Topic Graph represents groups of related content (communities) and the topic hierarchy they form. By sampling evidence from multiple topics, you are more likely to get evidence that reflects a broader range of themes than when you return only the nearest vectors. In library terms, when someone asks "What subject areas does this library cover?", the system does not gather only the area around the one book closest to the words of the question. It gathers evidence from the shelves of several subject areas. Internal processing, such as how many books to pick per shelf or in what order to examine the hierarchy, is not published. The processing order that the public documentation confirms is summarized in "Processing order of `retrieve`" after 2.3.

Among graph features, an Entity Graph follows the links between entities such as people, organizations and places (for example, which facility holds which designation). What a Topic Graph supports is cross-theme search, and no ontology is required. The `retrieve` in this notebook does not generate GQL, and it is normal that `graphSearch` does not appear in the activity log (2.8.1).

The idea of capturing the themes of a whole document set with communities is also covered in GraphRAG research. However, whether Foundry IQ's internal processing is the same is not published.

Source: [What is Foundry IQ?](https://learn.microsoft.com/azure/foundry/agents/concepts/what-is-foundry-iq) / [Agentic retrieval architecture and workflow](https://learn.microsoft.com/azure/search/agentic-retrieval-overview#architecture-and-workflow) / [Skipping reranking with `resultsProcessing`](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve#call-the-retrieve-action) / [Topic Graph overview](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph) / [Comparing Topic Graphs and Entity Graphs](https://learn.microsoft.com/azure/search/knowledge-graph-features-overview#compare-topic-graphs-and-entity-graphs) / Edge et al., [From Local to Global: A Graph RAG Approach to Query-Focused Summarization](https://arxiv.org/abs/2404.16130), 2024

### How the chunk index refers to the Topic Graph

The knowledge source searches the chunk index. When you use a Topic Graph, the vector field being searched refers indirectly to the topic index in the same Search service. The arrows in the diagram show configuration references, not the order of search processing.

```mermaid
flowchart TD
    KB["Knowledge base"] -->|"knowledgeSources"| KS["Knowledge source (searchIndex)"]
    KS -->|"searchIndexParameters.searchIndexName"| C["Chunk index"]
    C --> V["Vector field being searched"]
    V -->|"topicGraphProfile"| P["Topic profile in the same index"]
    P -->|"indexName"| T["Topic index<br/>graphProfile: definition of the graph structure"]
```

| Setting | Role |
|---|---|
| `graphProfile` | Defines the graph structure in the topic index. It is not the setting that points from the chunk index to the topic index. |
| `topicGraphProfile` | The profile name set on the vector field being searched. The service looks for the item with the same name in the chunk index's `vectorSearch.topicGraphProfiles` and refers to the topic index through its `indexName`. |
| `vectorSearchProfile` | The profile used for nearest-neighbor search and for vectorizing the query. The question is converted with the vectorizer of the same embedding model as at build time. The reference to the topic index is handled by the separate `topicGraphProfile`. |

When you enable topic sampling, Agentic Retrieval uses the **topic index** to select candidates. The text returned as evidence is taken from the **chunk index**. You do not need to specify the topic index directly in the knowledge source. What connects the two indexes is checked in "What connects the two indexes" after 2.3.

This notebook sets the knowledge source default to `balanced` (diversity factor fixed at `0.2`). In 2.5, `none` is the baseline that disables topic sampling. In 2.7, `diverse` is the setting where `diversityFactor` is changed per request in the range `0`-`1`. Changing the factor does not necessarily change the top sources. You check the differences by comparing actual search responses in 2.8.

**To check this in your own environment:** Specify the chunk index in `CONTENT_INDEX`, the registered topic index that corresponds to those chunks in `TOPIC_INDEX`, and the vector field being searched in `VECTOR_FIELD`, and check the reference relationship in 2.2. If the settings of the existing knowledge base and knowledge source match, you do not need to recreate them.

Source: [Microsoft Learn: Checking the Topic Graph reference](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#set-up-requests) / [Overriding topic selection per request](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#override-topic-selection-for-one-request) / [Settings and default values](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#review-settings-and-verification-limits)

## 1. Connection settings and the boundary of changes

### Azure prerequisites

| Item | What you need |
|---|---|
| Service | Foundry IQ / Azure AI Search that supports `2026-11-01-preview` and the graph features |
| Topic Graph | `ft-topic-src` built in Functional Test 2, and the topic index registered in chapter 9 of Functional Test 2. Use the same embedding model and deployment as at build time. |
| Running user | Search Index Data Reader to read. Search Service Contributor is also required for a new configuration or for changing index settings. If your existing permissions are sufficient, no additional grants are needed. |
| Search managed identity | Cognitive Services OpenAI User on Azure OpenAI. Being able to sign in as a user does not mean the service can connect to the model. |

This notebook does not create services or models, assign roles, upload documents, re-extract the graph or delete resources.

### Kernel dependencies

If a live cell raises an import error, or if the SDK is not `12.2.0b1`, run the following in the notebook kernel you selected. It pins the same versions as Functional Test 2.

```python
%pip install azure-search-documents==12.2.0b1 azure-identity python-dotenv
```

Using [.env.example](./.env.example) as a reference, prepare a `.env` in the same folder as this notebook, and sign in to the target tenant with, for example, the Azure CLI. Never write credentials or access tokens directly in cells. As in Functional Test 2, the settings in `.env` take priority.

### Environment variables and defaults

This notebook does not use a chat model, so `AZURE_OPENAI_DEPLOYMENT` is not needed.

| Environment variable | When it is needed | Default when unset |
|---|---|---|
| `AZURE_SEARCH_ENDPOINT` | Required when `RUN_AZURE = True` | None |
| `AZURE_OPENAI_ENDPOINT` | In 2.2, when the search field has no vectorizer and one must be added | None |
| `AZURE_OPENAI_EMBEDDING_DEPLOYMENT` | Same as above | None |
| `AZURE_OPENAI_EMBEDDING_MODEL` | Same as above. Set it when the deployment name and the model name differ. | The embedding deployment name (the same handling as in Functional Test 2, where the deployment name is used if `AZURE_OPENAI_MODEL` is unset) |
| `FOUNDRY_IQ_TOPIC_INDEX` | When you specify the registered topic index explicitly | Empty. The notebook looks for the record saved in 9.3 of Functional Test 2, or for an index whose name starts with `ft-published-topics-` (2.1). |

The names of the index, knowledge source and knowledge base that this notebook uses are derived from the same prefix as Functional Test 2, `FT = "ft-topic"`. To use different names, change `FT` in the settings cell.

| Target | Default name |
|---|---|
| Chunk index | `ft-topic-src` |
| Knowledge source | `ft-topic-agentic-ks` |
| Knowledge base | `ft-topic-agentic-kb` |

### Run switches

- `RUN_AZURE`: The default is `True`, which connects to Azure and runs searches. Set it to `False` to skip connecting. Embedding the question may incur charges.
- `ENABLE_TOPIC`: The default is `True`. Setting it to `False` skips Topic Graph searches and configuration changes.
- `CONFIRM_WRITES`: The default is `False`. Review the plan, and only if a change is needed, set it to `True` and rerun the settings cell. This setting approves creating the dedicated knowledge base and knowledge source, and changing the search settings of the verification index `ft-topic-src`. It does not overwrite an existing source or an existing knowledge base.

Source: [Topic Graph prerequisites](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#prerequisites) / [Permissions to search a knowledge base](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve#prerequisites) / [RBAC for Search](https://learn.microsoft.com/azure/search/search-security-rbac) / [Azure Identity authentication for local development](https://learn.microsoft.com/azure/developer/python/sdk/authentication/local-development-environment)

In [ ]:
from __future__ import annotations

import copy
import html
import json
import os
from pathlib import Path
from urllib.parse import urlsplit

from IPython.display import HTML, SVG, display


def show_json(value: object) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2))


def preview_text(value: object) -> object:
    return value[:200] + "..." if isinstance(value, str) and len(value) > 200 else value


def table(rows: list[dict]) -> None:
    if not rows:
        print("There are no rows to display.")
        return
    columns = list(dict.fromkeys(key for row in rows for key in row))
    header = "".join(f"<th>{html.escape(str(key))}</th>" for key in columns)
    body = []
    for row in rows:
        cells = []
        for key in columns:
            value = row.get(key, "")
            text = json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else str(value)
            cells.append(f"<td>{html.escape(text)}</td>")
        body.append("<tr>" + "".join(cells) + "</tr>")
    display(HTML("<table><thead><tr>" + header + "</tr></thead><tbody>" + "".join(body) + "</tbody></table>"))

### What to specify for `CONTENT_INDEX` and `TOPIC_INDEX`

**`CONTENT_INDEX` is where the evidence text is retrieved from, and `TOPIC_INDEX` is what is referenced to select evidence candidates by topic.** Using an internal manual as an analogy, the former holds the body text such as application procedures, and the latter holds topics such as "application" and "approval" together with the mapping to the text fragments that belong to each.

| Setting | Name to copy | Naming convention in this repository |
|---|---|---|
| `CONTENT_INDEX` | The name of the chunk index used as input when the Topic Graph was built. In [sections 1.2 and 4 of Functional Test 2](./functional-test-2-topic-graph.ipynb), the value of `SRC_INDEX` is passed to `search.sourceIndex`. | `<FT>-src`. `FT` is a prefix for the tests, and the default in Functional Test 2 is `ft-topic`. |
| `TOPIC_INDEX` | The index name used when the Topic Graph built from those chunks was registered in Azure AI Search. Copy it from the registration result in [section 9.3 of Functional Test 2](./functional-test-2-topic-graph.ipynb) or from `topic_index` in `.functional-test/published-resources.json`. | `<PREFIX>-topics-<first 12 digits of the hash>`. The default `PREFIX` is `ft-published`. The hash is computed from the Topic Graph schema and the registered documents. |

What you specify are **separate index names in the same Search service, corresponding to the input and the registration target of the same build**. The naming convention is only a landmark for the functional tests. Do not reconstruct the names; use the values from when you actually built and registered. The local output of Functional Test 2 cannot be specified as `TOPIC_INDEX`. If the Topic Graph is not yet registered, register it in Search first.

If the connection already exists, the knowledge source's `searchIndexParameters.searchIndexName` is `CONTENT_INDEX`. Follow the `topicGraphProfile` that the search-target vector field of the chunk index specifies, and check the `indexName` of the profile with the same name to find `TOPIC_INDEX`. [Microsoft Learn: Topic Graph reference settings](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#set-up-requests)

In [ ]:
RUN_AZURE = True
ENABLE_TOPIC = True
CONFIRM_WRITES = False
API_VERSION = "2026-11-01-preview"

FT = "ft-topic"  # Same prefix as Functional Test 2
NOTEBOOK_DIR = Path.cwd()

if RUN_AZURE:
    from importlib.metadata import version
    from azure.core import MatchConditions
    from azure.core.exceptions import HttpResponseError, ResourceNotFoundError
    from azure.core.pipeline import PipelineResponse
    from azure.identity import DefaultAzureCredential
    from azure.search.documents import SearchClient
    from azure.search.documents.indexes import SearchIndexClient
    from azure.search.documents.indexes.models import (
        KnowledgeBase, KnowledgeSource, KnowledgeSourceReference, SearchIndex, SearchIndexFieldReference,
        SearchIndexKnowledgeSource, SearchIndexKnowledgeSourceParameters,
    )
    from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
    from azure.search.documents.knowledgebases.models import (
        KnowledgeBaseRetrievalRequest, KnowledgeRetrievalMinimalReasoningEffort,
        KnowledgeRetrievalSemanticIntent, SearchIndexKnowledgeSourceParams,
    )
    from azure.search.documents.models import (
        BalancedVectorQueryTopicGraph, DiverseVectorQueryTopicGraph, NoneVectorQueryTopicGraph,
    )
    from dotenv import load_dotenv
    if version("azure-search-documents") != "12.2.0b1":
        raise RuntimeError("Install the same azure-search-documents==12.2.0b1 as Functional Test 2 in the kernel.")
    load_dotenv(NOTEBOOK_DIR / ".env", override=True)
    credential = DefaultAzureCredential()

SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT", "").rstrip("/")
MODEL_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT", "").rstrip("/")
EMBED_DEPLOYMENT = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "")
EMBED_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL", EMBED_DEPLOYMENT)
CONTENT_INDEX = f"{FT}-src"
VECTOR_FIELD = "snippet_vector"
CONTENT_FIELD = "snippet"
TOPIC_INDEX = os.getenv("FOUNDRY_IQ_TOPIC_INDEX", "")
TOPIC_SOURCE = f"{FT}-agentic-ks"
TOPIC_KB = f"{FT}-agentic-kb"

if RUN_AZURE:
    endpoint = urlsplit(SEARCH_ENDPOINT)
    if (endpoint.scheme != "https" or not endpoint.hostname or endpoint.username
            or endpoint.password or endpoint.path or endpoint.query or endpoint.fragment or "<" in SEARCH_ENDPOINT):
        raise ValueError("Set AZURE_SEARCH_ENDPOINT to the actual HTTPS service URL.")
    client_options = {"api_version": API_VERSION, "retry_total": 0, "connection_timeout": 10, "read_timeout": 90}
    index_client = SearchIndexClient(SEARCH_ENDPOINT, credential, **client_options)
    azure_clients: list[SearchIndexClient | SearchClient | KnowledgeBaseRetrievalClient] = [index_client]
    if ENABLE_TOPIC and not TOPIC_INDEX:
        state_path = Path(".functional-test/published-resources.json")
        if state_path.is_file():
            state = json.loads(state_path.read_text(encoding="utf-8"))
            if state["endpoint"].rstrip("/") != SEARCH_ENDPOINT or state["api_version"] != API_VERSION:
                raise ValueError("The saved record of Functional Test 2 differs from the current endpoint or API version. Check the registration target.")
            TOPIC_INDEX = state.get("topic_index", "")
        if not TOPIC_INDEX:
            candidates = sorted(name for name in index_client.list_index_names() if name.startswith("ft-published-topics-"))
            table([{"Topic Graph candidate": name} for name in candidates])
            if len(candidates) != 1:
                raise ValueError("There are zero or multiple candidates. Check the registration in chapter 9 of Functional Test 2 and set the registered name in FOUNDRY_IQ_TOPIC_INDEX.")
            TOPIC_INDEX = candidates[0]

table([{"Azure run": RUN_AZURE, "Topic run": ENABLE_TOPIC, "Write approval": CONFIRM_WRITES,
        "Topic index": TOPIC_INDEX or "Not specified"}])

### 1.1 SDK exception handling and safe creation

Authentication and transport are left to the SDK. Service errors stop at the SDK's `HttpResponseError`, and only a missing resource is handled as `ResourceNotFoundError`. A partial `retrieve` response, `206`, is detected with a response hook and stops execution. Because each client uses `retry_total = 0`, no automatic retries happen.

New resources use `MatchConditions.IfMissing` to prevent overwriting existing ones. If a resource already exists on rerun, it is reused after its settings are checked against the plan in this notebook. Defaults that the server adds are not compared.

Source: [Concurrency control in Search](https://learn.microsoft.com/azure/search/search-howto-concurrency) / [Knowledge Retrieval REST](https://learn.microsoft.com/rest/api/searchservice/knowledge-retrieval/retrieve?view=rest-searchservice-2026-11-01-preview)

In [ ]:
def reject_partial_response(response: PipelineResponse) -> None:
    if response.http_response.status_code == 206:
        raise HttpResponseError("Agentic Retrieval returned a partial response. Check the response body and the activity log.",
                                response=response.http_response)


def require_expected(actual: object, expected: object, path: str = "resource") -> None:
    if isinstance(expected, dict) and isinstance(actual, dict):
        for key, value in expected.items():
            require_expected(actual.get(key), value, f"{path}.{key}")
    elif isinstance(expected, list) and isinstance(actual, list) and len(actual) == len(expected):
        for index, value in enumerate(expected):
            require_expected(actual[index], value, f"{path}[{index}]")
    elif actual != expected:
        raise ValueError(f"The existing configuration differs from the plan: {path}. Do not overwrite it; use a different name or check the settings.")


def create_or_reuse(definition: KnowledgeBase | SearchIndexKnowledgeSource) -> KnowledgeBase | KnowledgeSource:
    if not RUN_AZURE:
        raise RuntimeError("Azure calls are disabled. Check RUN_AZURE.")
    try:
        if isinstance(definition, KnowledgeBase):
            existing = index_client.get_knowledge_base(definition.name)
        else:
            existing = index_client.get_knowledge_source(definition.name)
    except ResourceNotFoundError:
        print("Not yet created:", definition.name)
        if not CONFIRM_WRITES:
            raise RuntimeError("Review the plan, set CONFIRM_WRITES = True and rerun from this cell.") from None
        if isinstance(definition, KnowledgeBase):
            return index_client.create_or_update_knowledge_base(definition, match_condition=MatchConditions.IfMissing)
        return index_client.create_or_update_knowledge_source(definition, match_condition=MatchConditions.IfMissing)
    require_expected(existing.as_dict(), definition.as_dict())
    print("Settings verified, reusing:", definition.name)
    return existing

## 2. Search for the themes of a document set with a Topic Graph

### 2.1 Check the content and topic definitions and the registered data

If `FOUNDRY_IQ_TOPIC_INDEX` is empty, the registration target is read from `.functional-test/published-resources.json`, which 9.3 of Functional Test 2 saved. If there is no record, the SDK looks for an index whose name starts with `ft-published-topics-`, as in 9.1 of Functional Test 2. If there is exactly one candidate, it is reused, and if there are zero or multiple candidates, the notebook stops. Specify a different registration target explicitly in the connection settings.

Check that both indexes contain data. The notebook also checks the embedding dimensions, but **the same dimensions alone do not prove that the vectors were generated by the same model.** Set the embedding deployment and model used at build time.

Source: [Topic Graph prerequisites and definition check](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#prerequisites)

In [ ]:
if RUN_AZURE and ENABLE_TOPIC:
    content_schema = index_client.get_index(CONTENT_INDEX).as_dict()
    topic_schema = index_client.get_index(TOPIC_INDEX).as_dict()
    content_vector = next(field for field in content_schema["fields"] if field["name"] == VECTOR_FIELD)
    topic_dimensions = {field["dimensions"] for field in topic_schema["fields"] if field.get("dimensions")}
    if content_vector["dimensions"] not in topic_dimensions:
        raise ValueError("The vector dimensions of the topic index and the content index do not match. Check the settings used at build time.")
    for schema in (content_schema, topic_schema):
        key_name = next(field["name"] for field in schema["fields"] if field.get("key"))
        with SearchClient(SEARCH_ENDPOINT, schema["name"], credential, **client_options) as source_client:
            sample = source_client.search("*", select=[key_name], top=1, include_total_count=True)
            count = sample.get_count()
            if count is None:
                raise RuntimeError("Cannot get the document count. Check the include_total_count response.")
            table([{"Index": schema["name"], "Document count": count, "Example keys": list(sample)}])
        if count == 0:
            raise ValueError("The index is empty. Complete the build and the chapter 9 registration of Functional Test 2.")
    show_json(content_vector)
else:
    print("Not run: check of the Topic Graph definition and data. RUN_AZURE and ENABLE_TOPIC are required.")

### 2.2 Plan for connecting the Topic Graph to the search field

**Purpose of this section:** The Topic Graph searches in 2.5-2.7 (`none`, `balanced`, `diverse`) assume that the vector field of the chunk index being searched is connected to the topic index. The official prerequisites also require that the vector field used for search be linked to the prepared topic index, and when the link is missing, they direct you to check `topicGraphProfile` on the search-target field. Because the question text is converted to a vector, that field also needs a vectorizer.

**Relationship to the official procedure:** The Microsoft Learn procedure assumes an already-connected index and does not describe how to add the connection later. As part of that procedure, it also asks you not to send a replacement index definition. The setup for the case where no connection exists is handled by the TopicGraph setup of the Foundry IQ skills plugin (`plan-retrieval` → `configure-retrieval` → `verify-retrieval`). According to the plugin's documentation, it configures the topic connection on the search-target vector field and adjusts the existing knowledge source, with approval. This section is this notebook's implementation that makes the same kind of change with the SDK, and it is not the plugin's own processing. The connection structure (the field's `topicGraphProfile` and `vectorSearch.topicGraphProfiles`) follows the official REST definition example, and nothing is changed if the connection and the vectorizer are already in place.

Chapter 9 of Functional Test 2 registers the topics, but it does not set the search field's `topicGraphProfile` or the vectorizer that embeds the question. Here the notebook copies the existing definition of the verification index `ft-topic-src` and adds only the required search settings. **The existing field's `vectorSearchProfile` reference and the search algorithm are not changed.**

If no connection is set, a topic profile is added. If there is no vectorizer, a vectorizer for the same embedding model is added, and its reference is set on the existing profile. An existing vectorizer is reused. If a different Topic Graph is already connected, the notebook stops. This automatic setup is limited to `ft-topic-src`. Do not rename it and apply it to a production index.

The ETag from when the definition was retrieved is set as the `e_tag` of the `SearchIndex` used for the update, and the update uses `MatchConditions.IfNotModified`. If another change has been made in the meantime, the notebook stops with `412`. The definition before the update is kept in the kernel in `content_schema`, but it is not saved to a file, because it might contain credentials.

Source: [Topic Graph prerequisites](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#prerequisites) / [Topic Graph troubleshooting](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#troubleshooting) / [Checking the Topic Graph connection](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#set-up-requests) / [Foundry IQ skills: retrieval setup procedure (retrieval-setup)](https://github.com/microsoft/azure-skills/blob/7847020998043c6231929b364c35393b7cdab270/.github/plugins/foundry-iq-skills/skills/foundry-iq/workflows/graph/topic-graph-build/references/retrieval-setup.md) / [Foundry IQ skills: prerequisite setup for topic diversity](https://github.com/microsoft/azure-skills/blob/7847020998043c6231929b364c35393b7cdab270/.github/plugins/foundry-iq-skills/graph/apps-host/standalone-topic-diversity.md#native-setup-prerequisite) / [Official REST definition example for the topic connection](https://github.com/Azure/azure-rest-api-specs/blob/be93aca8c35ada896dd7fd6c7f65b1a9f766e84c/specification/search/data-plane/Search/preview/2026-11-01-preview/examples/SearchServiceCreateOrUpdateIndexWithTopicGraphs.json) / [Adding a vectorizer to an existing index](https://learn.microsoft.com/azure/search/vector-search-how-to-configure-vectorizer#define-a-vectorizer-programmatically) / [Update restrictions for vectorized fields](https://learn.microsoft.com/azure/search/vector-search-how-to-create-index#update-a-vector-index) / [Concurrency control](https://learn.microsoft.com/azure/search/search-howto-concurrency)

In [ ]:
if RUN_AZURE and ENABLE_TOPIC:
    topic_index_plan = None
    vector_search = content_schema["vectorSearch"]
    old_profile = next(profile for profile in vector_search["profiles"] if profile["name"] == content_vector["vectorSearchProfile"])
    bound_name = content_vector.get("topicGraphProfile")
    bound_profile = next((profile for profile in vector_search.get("topicGraphProfiles", []) if profile["name"] == bound_name), None)
    existing_vectorizer = next((item for item in vector_search.get("vectorizers", []) if item["name"] == old_profile.get("vectorizer")), None)
    if bound_name and (bound_profile is None or bound_profile["indexName"] != TOPIC_INDEX):
        raise ValueError("The search field already has a different topic connection. Do not overwrite it; check the settings.")
    if bound_profile is not None and existing_vectorizer is not None:
        print("The Topic Graph and the vectorizer are already connected. No update.")
    else:

        topic_index_plan = {key: copy.deepcopy(value) for key, value in content_schema.items() if not key.startswith("@odata.")}
        planned_search = topic_index_plan["vectorSearch"]
        planned_profile = next(profile for profile in planned_search["profiles"] if profile["name"] == old_profile["name"])
        additions = {}
        if bound_profile is None:
            additions["topicGraphProfiles"] = {"name": "iq-notebook-topics", "indexName": TOPIC_INDEX,
                                                "defaultKind": "balanced", "defaultMinDiversityPoolSize": 1000}
        if existing_vectorizer is None:
            if not all((MODEL_ENDPOINT, EMBED_DEPLOYMENT, EMBED_MODEL)) or any("<" in value for value in (MODEL_ENDPOINT, EMBED_DEPLOYMENT, EMBED_MODEL)):
                raise ValueError("Set the same embedding deployment and the actual model name as Functional Test 2.")
            other_fields = [field["name"] for field in content_schema["fields"]
                            if field.get("vectorSearchProfile") == old_profile["name"] and field["name"] != VECTOR_FIELD]
            if other_fields:
                raise ValueError(f"The profile is also used by other fields: {other_fields}. Stopping the automatic change.")
            additions["vectorizers"] = {"name": "iq-notebook-vectorizer", "kind": "azureOpenAI", "azureOpenAIParameters": {
                "resourceUri": MODEL_ENDPOINT, "deploymentId": EMBED_DEPLOYMENT, "modelName": EMBED_MODEL}}
            planned_profile["vectorizer"] = "iq-notebook-vectorizer"
        for collection, addition in additions.items():
            items = planned_search.setdefault(collection, [])
            existing = next((item for item in items if item["name"] == addition["name"]), None)
            if existing is None:
                items.append(addition)
            else:
                require_expected(existing, addition)
        planned_field = next(field for field in topic_index_plan["fields"] if field["name"] == VECTOR_FIELD)
        planned_field["topicGraphProfile"] = bound_name or "iq-notebook-topics"
        show_json({"Target of change": CONTENT_INDEX, "Changed field": planned_field,
                   "Query settings for the existing profile": planned_profile, "Search settings to add": additions})
else:
    print("Not run: plan for changing the Topic Graph connection.")

In [ ]:
if RUN_AZURE and ENABLE_TOPIC:
    if topic_index_plan is not None:
        if not CONFIRM_WRITES:
            raise RuntimeError("Review the embedding model, the target and the change plan, then set CONFIRM_WRITES = True.")
        topic_index_model = SearchIndex(topic_index_plan)
        topic_index_model.e_tag = content_schema["@odata.etag"]
        index_client.create_or_update_index(topic_index_model, match_condition=MatchConditions.IfNotModified)
    effective_schema = index_client.get_index(CONTENT_INDEX).as_dict()
    effective_field = next(field for field in effective_schema["fields"] if field["name"] == VECTOR_FIELD)
    effective_search = effective_schema["vectorSearch"]
    effective_topic = next(profile for profile in effective_search["topicGraphProfiles"] if profile["name"] == effective_field["topicGraphProfile"])
    effective_vector = next(profile for profile in effective_search["profiles"] if profile["name"] == effective_field["vectorSearchProfile"])
    if effective_topic["indexName"] != TOPIC_INDEX:
        raise ValueError("The read-back Topic Graph target differs from the plan.")
    effective_vectorizer = next(item for item in effective_search["vectorizers"] if item["name"] == effective_vector["vectorizer"])
    show_json({"Topic connection": effective_topic, "Vectorizer name": effective_vectorizer["name"]})
else:
    print("Not run: updating the verification index settings and reading them back.")

### 2.3 Create the topic search source and the knowledge base

The field to search is `snippet_vector`, and the fields returned as the original text are `uid`, `snippet` and `blob_url`. The source default is `balanced`, which is changed per request later. The knowledge base that uses `minimal` has no chat model configured for search planning.

Only the dedicated source and KB with the default names are created, and existing configurations that differ are not overwritten. `uid` is included in the source's `sourceDataFields` so that results can be compared by the same chunk ID, not by a reference ID that can change from search to search.

Source: [Creating a Topic Graph source](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#create-the-knowledge-source) / [Attaching the source to a knowledge base](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#attach-the-source-to-a-knowledge-base)

In [ ]:
if RUN_AZURE and ENABLE_TOPIC:
    topic_source_plan = SearchIndexKnowledgeSource(
        name=TOPIC_SOURCE, results_processing="none",
        description="agentic-retrieval-graphs notebook: topic retrieval",
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=CONTENT_INDEX, search_fields=[SearchIndexFieldReference(name=VECTOR_FIELD)],
            source_data_fields=[SearchIndexFieldReference(name=name) for name in ("uid", CONTENT_FIELD, "blob_url")],
            topic_graph=BalancedVectorQueryTopicGraph(min_diversity_pool_size=1000),
        ),
    )
    show_json(topic_source_plan.as_dict())
    topic_source_definition = create_or_reuse(topic_source_plan)
    require_expected(topic_source_definition.as_dict(), topic_source_plan.as_dict())
else:
    print("Not run: creating and verifying the Topic Graph search source.")

In [ ]:
if RUN_AZURE and ENABLE_TOPIC:
    topic_kb_plan = KnowledgeBase(
        name=TOPIC_KB, description="agentic-retrieval-graphs notebook: topic retrieval",
        knowledge_sources=[KnowledgeSourceReference(name=TOPIC_SOURCE)],
        retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(), output_mode="extractiveData",
    )
    show_json(topic_kb_plan.as_dict())
    topic_kb_definition = create_or_reuse(topic_kb_plan)
    require_expected(topic_kb_definition.as_dict(), topic_kb_plan.as_dict())
    topic_kb_client = KnowledgeBaseRetrievalClient(SEARCH_ENDPOINT, credential, knowledge_base_name=TOPIC_KB, **client_options)
    azure_clients.append(topic_kb_client)
else:
    print("Not run: creating and verifying the Topic Graph search knowledge base.")

### Processing order of `retrieve`: from candidate retrieval to topic sampling

**The key point is the two-stage approach: first gather candidates broadly with vector search, then reselect evidence using topics.** The order that the public specification confirms is that vector-search candidate retrieval comes before topic sampling. The following is the flow that can be confirmed from official documentation for `2026-11-01-preview`, which this notebook uses. In the diagram, boxes with blue solid borders are steps documented in official sources, and boxes with orange dashed borders are steps whose existence is confirmed but whose internal behavior is not published.

```mermaid
flowchart TD
    Q["① Caller<br/>Passes one question (intents) and a topicGraph setting to retrieve"]
    P["② Planning (minimal)<br/>No model planning; searches all target knowledge sources"]
    V["③ Vectorize the question<br/>Uses the vectorizer of the search field"]
    C["④ Vector search<br/>balanced and diverse retrieve at least minDiversityPoolSize<br/>candidates before sampling (default 1000)"]
    S["⑤ Topic sampling<br/>Uses representative topic vectors to select evidence from multiple themes<br/>balanced: factor fixed at 0.2 / diverse: factor 0-1 specified"]
    N["⑤' none<br/>No sampling; uses the regular vector search results"]
    R["⑥ Response (extractiveData)<br/>Excerpts response / sources references / activity log activity"]
    T[("Topic index<br/>Topic IDs and centroid vectors")]
    K[("Chunk index ft-topic-src<br/>Chunk text and snippet_vector")]
    Q --> P --> V --> C
    K -->|"search target"| C
    C -->|"balanced / diverse"| S --> R
    C -->|"none"| N --> R
    K -.->|"fetch the excerpt text"| R
    T -.->|"referenced via indexName of topicGraphProfile"| S
    classDef known fill:#e8f2ff,stroke:#3976b8,stroke-width:1.5px;
    classDef opaque fill:#fff7e6,stroke:#d97706,stroke-width:2px,stroke-dasharray:5 4;
    class Q,P,V,C,N,R,T,K known;
    class S opaque;
```

The stage at which the topic index is referenced is not published. It is drawn with a dotted line, tied to the sampling that is described as using representative topic vectors.

#### Two indexes are involved

| Index | Name in this notebook | Contents | How it is used |
|---|---|---|---|
| Chunk index | `ft-topic-src` (`CONTENT_INDEX`) | The text and the vectors (`snippet_vector`) of the chunks that the documents were split into | The search target that the knowledge source specifies directly. Search runs against it in ④, and the text of the excerpts returned in ⑥ is also taken from it. |
| Topic index | `ft-published-topics-<hash>` (`TOPIC_INDEX`) | Topic IDs and centroid vectors. It also has the definition of the graph structure that represents the hierarchy (`graphProfile`). | The knowledge source does not specify it directly. It is referenced through `indexName` from the `topicGraphProfile` of the chunk index's vector field. |

A knowledge source and a knowledge base are not indexes. They are settings that define the search target and the entry point.

#### What happens at each stage

| Stage | What happens | Specification in this notebook |
|---|---|---|
| ① Request | Passes the question and `topicGraph` to `retrieve`. The request's `topicGraph` overrides the setting saved in the knowledge source for that request only. | One question in `intents`. `minimal` does not accept `messages`. Switch `topicGraph.kind` among `none`, `balanced` and `diverse`. |
| ② Planning | The usual `minimal` skips model planning and searches all target knowledge sources. | The target is the one dedicated knowledge source. No LLM is used. |
| ③ Vectorization | Converts the question text to a vector with the search field's vectorizer. | Uses the vectorizer configured on `snippet_vector`. |
| ④ Candidate retrieval | Runs vector search against the chunk index and gathers the nearest chunks as candidates. With `balanced` and `diverse`, at least `minDiversityPoolSize` candidates are retrieved before sampling. | `minDiversityPoolSize = 1000` (the allowed range is 250-10000). Not specified for `none`. |
| ⑤ Topic sampling | Uses representative topic vectors to select evidence from multiple themes. `balanced` combines similarity and diversity, and its diversity factor is fixed at 0.2. `diverse` takes `diversityFactor` (0-1) per request. `none` skips this stage. | The `diverse` factor is 0.1. |
| ⑥ Response | With `extractiveData`, no answer is generated. It returns chunk excerpts (`response`), sources (`references`) and the activity log (`activity`). | References, reference source data and the activity log are all enabled. |

#### Where each setting comes from

| Setting | How it is determined |
|---|---|
| `topicGraph.kind` | Uses the request's `topicGraph` if present, otherwise the value saved in the knowledge source. |
| `diversityFactor` (`diverse` only) | Specified in the request. If omitted, the topic profile's value is used, and if that is absent, 0.2. `balanced` is fixed at 0.2 and is independent of the profile value. |
| `minDiversityPoolSize` (`balanced`, `diverse`) | Specified in the request. If omitted, the topic profile's value is used, and if that is absent, 1000. |

This notebook saves `balanced` (1000 candidates) in the knowledge source and overrides it in every request of 2.5-2.7 by stating `kind` and the candidate count explicitly. Official documentation does not let us confirm how the topic profile's `defaultKind` is used in searches through a knowledge base, so the notebook does not rely on the profile defaults.

Source: [Topic Graph retrieval requests and settings](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph) / [`topicGraph` definition in the Retrieve API](https://learn.microsoft.com/rest/api/searchservice/knowledge-retrieval/retrieve?view=rest-searchservice-2026-11-01-preview#balancedvectorquerytopicgraph) / [`topicGraphProfile` in the index definition](https://learn.microsoft.com/rest/api/searchservice/indexes/create-or-update?view=rest-searchservice-2026-11-01-preview) / [Topic Graph overview (representative vectors and centroid vectors)](https://learn.microsoft.com/azure/search/knowledge-graph-features-overview#compare-topic-graphs-and-entity-graphs) / [Behavior of the `minimal` reasoning effort](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-set-retrieval-reasoning-effort) / [Retrieval procedure and response](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve) / [Vectorization at query time](https://learn.microsoft.com/azure/search/vector-search-how-to-configure-vectorizer)

### What connects the two indexes

**What connects the two indexes is a reference by index name in the configuration. A document key such as a chunk's `uid` is not used as a join key.**

| Layer | Connection point | Basis |
|---|---|---|
| Configuration | The `topicGraphProfile` (profile name) of the chunk index's vector field points to the item with the same name in `vectorSearch.topicGraphProfiles` of the same index, and its `indexName` is the name of the topic index. | Described in the official REST definition and the prerequisites. |
| Data | Official documentation explains that the topic index holds topic IDs and centroid vectors. It does not describe a join key that links a chunk's `uid` to a topic. | Official documentation. The next cell checks the actual contents. |

The next cell is read-only and checks two points. It uses stored vectors, so it does not call the embedding model.

1. The number of documents in the topic index that have a value in the fields that hold chunk IDs (`sourceChunkIdField` and `sourceParentIdField` in the graph definition).
2. The result of searching the topic index's vector field with the stored vector of one chunk.

In the topic index built from the default input of Functional Test 2, there were 0 documents that held chunk IDs. The vector field holds the centroid vectors of the topic documents that have the predicate `HasCentroidVector`, and the topics close to a chunk's vector are returned. In other words, with this data the correspondence between chunks and topics can be expressed by vector proximity rather than by ID. With other data, the counts and names will differ.

**How `retrieve` uses this correspondence internally is not published.** This cell checks how the contents of the two indexes can correspond, and it does not reproduce the sampling procedure.

Source: [`topicGraphProfile` in the index definition](https://learn.microsoft.com/rest/api/searchservice/indexes/create-or-update?view=rest-searchservice-2026-11-01-preview) / [Topic Graph prerequisites](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#prerequisites) / [Topic Graph overview (topic IDs and centroid vectors)](https://learn.microsoft.com/azure/search/knowledge-graph-features-overview#compare-topic-graphs-and-entity-graphs) / [Vector queries](https://learn.microsoft.com/azure/search/vector-search-how-to-query)

In [ ]:
if RUN_AZURE and ENABLE_TOPIC:
    from azure.search.documents.models import VectorizedQuery

    graph_fields = topic_schema["graphProfile"]["fields"]
    topic_vector_field = next(field["name"] for field in topic_schema["fields"] if field.get("dimensions"))
    with SearchClient(SEARCH_ENDPOINT, CONTENT_INDEX, credential, **client_options) as content_client:
        probe = next(iter(content_client.search("*", select=["uid", CONTENT_FIELD, VECTOR_FIELD], top=1)))
    print("Chunk used for the search:", probe["uid"], preview_text(" ".join(probe[CONTENT_FIELD].split())))
    with SearchClient(SEARCH_ENDPOINT, TOPIC_INDEX, credential, **client_options) as topic_client:
        id_counts = []
        for label, key in (("Chunk ID", "sourceChunkIdField"), ("Parent document ID", "sourceParentIdField")):
            found = topic_client.search("*", filter=f"{graph_fields[key]} ne null", top=0, include_total_count=True)
            id_counts.append({"Value checked": label, "Field name": graph_fields[key], "Topic documents with a value": found.get_count()})
        table(id_counts)
        nearest = topic_client.search(
            search_text=None, top=5,
            select=[graph_fields["subjectIdField"], graph_fields["subjectIdDisplayNameField"], graph_fields["predicateField"]],
            vector_queries=[VectorizedQuery(vector=probe[VECTOR_FIELD], k_nearest_neighbors=5, fields=topic_vector_field)],
        )
        table([{"Similarity": round(hit["@search.score"], 4), "Topic ID": hit[graph_fields["subjectIdField"]],
                "Topic name": hit[graph_fields["subjectIdDisplayNameField"]], "Predicate": hit[graph_fields["predicateField"]]}
               for hit in nearest])
else:
    print("Not run: checking the correspondence between the chunk index and the topic index. RUN_AZURE and ENABLE_TOPIC are required.")

### 2.4 Change only the selection method with the same question

| `topicGraph.kind` | Meaning | Allowed values |
|---|---|---|
| `none` | Disables topic sampling | Do not specify diversity parameters |
| `balanced` | Fixed setting with diversity factor `0.2` | `minDiversityPoolSize` only |
| `diverse` | Specifies the diversity factor per request | `diversityFactor`: 0-1, `minDiversityPoolSize`: 250-10000 |

In the preview, the knowledge base does not look at the question and choose `diverse` automatically. You specify it explicitly in the request.

**The only thing that changes across the three methods is `topicGraph`.** The question, the knowledge source, the reasoning effort `minimal` and the output mode `extractiveData` are common, and the three requests are built with the same function `topic_request`. To compare the diversity settings under identical conditions, the reasoning effort is fixed to `minimal`. With `low` or `medium`, LLM search planning is added, and differences other than the method get mixed into the results. For `minimal`, you pass `intents`, not `messages`. Each of the following three cells runs **one live search**.

Source: [Per-request Topic Graph settings](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#override-topic-selection-for-one-request) / [Published setting ranges](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#review-settings-and-verification-limits) / [Reasoning effort](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-set-retrieval-reasoning-effort)

In [ ]:
TOPIC_QUESTION = "What are the main themes across the whole document set, covering history, buildings, annual events and tourism?"
topic_results = {}


def topic_request(question: str, kind: str, diversity_factor: float = 0.1, minimum_pool: int = 1000) -> KnowledgeBaseRetrievalRequest:
    if not question.strip() or kind not in ("none", "balanced", "diverse"):
        raise ValueError("Check the question and topicGraph.kind.")
    if kind != "none":
        if not 250 <= minimum_pool <= 10000:
            raise ValueError("minDiversityPoolSize must be in the range 250-10000.")
    if kind == "diverse":
        if not 0 <= diversity_factor <= 1:
            raise ValueError("diversityFactor must be in the range 0-1.")
    topic_parameters: NoneVectorQueryTopicGraph | BalancedVectorQueryTopicGraph | DiverseVectorQueryTopicGraph
    if kind == "none":
        topic_parameters = NoneVectorQueryTopicGraph()
    elif kind == "balanced":
        topic_parameters = BalancedVectorQueryTopicGraph(min_diversity_pool_size=minimum_pool)
    else:
        topic_parameters = DiverseVectorQueryTopicGraph(diversity_factor=diversity_factor, min_diversity_pool_size=minimum_pool)
    return KnowledgeBaseRetrievalRequest(
        intents=[KnowledgeRetrievalSemanticIntent(search=question)],
        retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(), output_mode="extractiveData", include_activity=True,
        knowledge_source_params=[SearchIndexKnowledgeSourceParams(
            knowledge_source_name=TOPIC_SOURCE, include_references=True, include_reference_source_data=True,
            topic_graph=topic_parameters)],
    )

### 2.4.1 Display the search results in a readable form

The notebook expands the JSON of the extracted result inside `response[].content[].text` and matches it to `references` by reference ID. The displayed text is the fragment that the search response actually returned. It is not replaced with the long full text on the reference side.

Sections 2.5-2.7 display in the order **question and search settings → counts → excerpt and source for each document**. To compare across methods, use the comparison table in 2.8. The search cells and the display cells are separate, so rerunning only a display cell after retrieval does not search again. Excerpts show the first 200 characters, and the display count is 5 by default. To see more fragments, call it as in `display_topic_result(topic_results["none"], "none", limit=10)`.

The original responses are kept in `topic_results`. Long chunk IDs can be checked in the collapsed identifier details. If the source is a URL, a link is shown, and if it is a file name, the name is shown. When a reference or source is missing, "Not recorded" is displayed, and the source is not guessed from the content.

Source: [Structure of extracted results, activity and references](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve?pivots=python#review-the-response)

In [ ]:
TOPIC_MODE_LABELS = {"none": "No topic selection", "balanced": "Fixed-factor selection", "diverse": "Specified-factor selection"}


def topic_passages(result: dict) -> list[dict]:
    references = {str(reference["id"]): reference for reference in result.get("references", [])}
    passages = []
    for message in result["response"]:
        for part in message.get("content", []):
            if part.get("type") != "text":
                print("Content type not handled by this display:", part.get("type"))
                continue
            text = part.get("text")
            if not isinstance(text, str):
                raise ValueError("The text in the search response is not a string.")
            items = json.loads(text) if text.lstrip().startswith(("[", "{")) else [{"content": text}]
            if isinstance(items, dict):
                items = [items]
            if not isinstance(items, list):
                raise ValueError("The extracted-result JSON must be an array or an object.")
            for item in items:
                if not isinstance(item, dict) or not isinstance(item.get("content"), str):
                    raise ValueError("The extracted result has no string content.")
                reference_id = item.get("ref_id")
                reference = references.get(str(reference_id), {})
                source_data = reference.get("sourceData")
                source_data = source_data if isinstance(source_data, dict) else {}
                body = item["content"]
                first_line, separator, remainder = body.partition("\n")
                title = item.get("title")
                if separator and first_line.strip().lower().endswith(".txt"):
                    title = title or first_line.strip()
                    body = remainder
                passages.append({"Reference ID": reference_id, "Document": title or "Search result",
                                 "Source text": body, "Source URL": source_data.get("blob_url"),
                                 "uid": source_data.get("uid"), "activity ID": reference.get("activitySource")})
    return passages


def display_topic_request(request: KnowledgeBaseRetrievalRequest) -> None:
    payload = request.as_dict()
    parameters = payload["knowledgeSourceParams"][0]["topicGraph"]
    table([{"Question": payload["intents"][0]["search"], "Method": parameters["kind"],
            "Topic settings": parameters, "Search source": payload["knowledgeSourceParams"][0]["knowledgeSourceName"]}])


def topic_excerpt(body: str) -> str:
    return str(preview_text(" ".join(body.split())))


def topic_source_link(source: object) -> str:
    if source is None:
        return "Source not recorded"
    escaped = html.escape(str(source))
    parsed_url = urlsplit(source) if isinstance(source, str) else None
    if parsed_url is not None and parsed_url.scheme in ("https", "http") and parsed_url.hostname:
        return f'<a href="{escaped}" target="_blank" rel="noopener noreferrer">Open source</a>'
    return f"Source: {escaped}"


def display_topic_result(result: dict, kind: str, limit: int = 5) -> None:
    if limit < 1:
        raise ValueError("The display limit must be 1 or more.")
    passages = topic_passages(result)
    print(f"{TOPIC_MODE_LABELS[kind]} ({kind})")
    table([{"Fragments returned": len(passages), "Source documents (retrieved)": len({row["Source URL"] for row in passages if row["Source URL"]}),
            "Displayed": min(limit, len(passages)), "Source not recorded": sum(row["Source URL"] is None for row in passages)}])
    if not passages:
        print("No text was returned. Check the search settings and the activity log.")
    for row in passages[:limit]:
        excerpt = topic_excerpt(row["Source text"])
        reference_id = row["Reference ID"] if row["Reference ID"] is not None else "Not recorded"
        source_link = topic_source_link(row["Source URL"])
        identifiers = {key: value if value is not None else "Not recorded" for key, value in {"Chunk ID": row["uid"], "activity ID": row["activity ID"]}.items()}
        display(HTML(f'<section style="border:1px solid #ccd5df;border-radius:8px;padding:12px;margin:8px 0">'
                     f'<strong>Reference {html.escape(str(reference_id))}: {html.escape(str(row["Document"]))}</strong>'
                     f'<p style="line-height:1.7">{html.escape(excerpt)}</p><p>{source_link}</p>'
                     f'<details><summary>Reference identifiers</summary><pre>{html.escape(json.dumps(identifiers, ensure_ascii=False, indent=2))}</pre></details></section>'))
    if len(passages) > limit:
        print(f"Showing the first {limit} of {len(passages)}. Increase limit to display the remaining excerpts.")


def display_topic_rank_table(passages_by_mode: dict[str, list[dict]], limit: int = 5) -> None:
    if limit < 1:
        raise ValueError("The display limit must be 1 or more.")
    count = min(limit, max((len(rows) for rows in passages_by_mode.values()), default=0))
    if not count:
        print("There are no returned fragments to show in the comparison table.")
        return
    header = '<th scope="col" style="width:4em">Return order</th>' + "".join(
        f'<th scope="col">{kind}</th>' for kind in TOPIC_MODE_LABELS)
    body = []
    for index in range(count):
        cells = [f'<th scope="row" style="vertical-align:top">{index + 1}</th>']
        for kind in TOPIC_MODE_LABELS:
            passages = passages_by_mode.get(kind)
            if passages is None:
                entry = "Not run"
            elif index >= len(passages):
                entry = "None"
            else:
                row = passages[index]
                title = html.escape(str(row["Document"]))
                excerpt = html.escape(topic_excerpt(row["Source text"]))
                entry = f'<strong>{title}</strong><br>{excerpt}<br>{topic_source_link(row["Source URL"])}'
            cells.append(f'<td style="text-align:left;vertical-align:top;overflow-wrap:anywhere">{entry}</td>')
        body.append("<tr>" + "".join(cells) + "</tr>")
    display(HTML(f'<table style="width:100%;table-layout:fixed">'
                 f'<caption>Comparison of the return order (up to the first {limit} of each method)</caption>'
                 f'<thead><tr>{header}</tr></thead><tbody>{"".join(body)}</tbody></table>'))

### 2.5 Baseline: no topic selection

`none` is a comparison request that does not use the Topic Graph.

In [ ]:
if RUN_AZURE and ENABLE_TOPIC:
    none_payload = topic_request(TOPIC_QUESTION, "none")
    display_topic_request(none_payload)
    topic_results["none"] = topic_kb_client.retrieve(none_payload, raw_response_hook=reject_partial_response).as_dict()
    print("Retrieved the result for none. The next display cell shows the excerpts and sources.")
else:
    print("Not run: live search without topic selection.")

In [ ]:
if RUN_AZURE and ENABLE_TOPIC:
    if "none" not in topic_results:
        raise RuntimeError("Run the search cell of 2.5 first.")
    display_topic_result(topic_results["none"], "none")
else:
    print("Not run: display of the none result.")

### 2.6 Search with a fixed diversity factor

`balanced` does not take `diversityFactor` and uses the fixed value `0.2`.

In [ ]:
if RUN_AZURE and ENABLE_TOPIC:
    balanced_payload = topic_request(TOPIC_QUESTION, "balanced")
    display_topic_request(balanced_payload)
    topic_results["balanced"] = topic_kb_client.retrieve(balanced_payload, raw_response_hook=reject_partial_response).as_dict()
    print("Retrieved the result for balanced. The next display cell shows the excerpts and sources.")
else:
    print("Not run: live search with balanced.")

In [ ]:
if RUN_AZURE and ENABLE_TOPIC:
    if "balanced" not in topic_results:
        raise RuntimeError("Run the search cell of 2.6 first.")
    display_topic_result(topic_results["balanced"], "balanced")
else:
    print("Not run: display of the balanced result.")

### 2.7 Search with an explicit diversity factor

The notebook sets `diversityFactor = 0.1`, the same as the official example, in the request. It does not assume that this value is optimal for this document set.

In [ ]:
if RUN_AZURE and ENABLE_TOPIC:
    diverse_payload = topic_request(TOPIC_QUESTION, "diverse", diversity_factor=0.1)
    display_topic_request(diverse_payload)
    topic_results["diverse"] = topic_kb_client.retrieve(diverse_payload, raw_response_hook=reject_partial_response).as_dict()
    print("Retrieved the result for diverse. The next display cell shows the excerpts and sources.")
else:
    print("Not run: live search with diverse.")

In [ ]:
if RUN_AZURE and ENABLE_TOPIC:
    if "diverse" not in topic_results:
        raise RuntimeError("Run the search cell of 2.7 first.")
    display_topic_result(topic_results["diverse"], "diverse")
else:
    print("Not run: display of the diverse result.")

### 2.8 Compare the search results for each `kind` side by side

From the responses retrieved in 2.5-2.7, the notebook tallies the number of returned fragments and the number of source documents, and puts `none`, `balanced` and `diverse` in columns so that you can compare document names, excerpts and sources. By default, the first 5 of each method are lined up. To raise this to 10, change the call below to `display_topic_rank_table(passages_by_mode, limit=10)`. Each row shows the same return position. It does not mean the same document or chunk.

The table can be shown even when only some of the methods have been searched. "Not run" marks a method that has no response, and "None" marks a return position that has no fragment. No new search happens at display time. If you change the question, search again with the same question for the methods you compare.

If `none` has been searched, you can also see the number of shared chunks and the fragments that are not in the baseline. The comparison key is `uid`, not the reference ID, which can change from search to search. A `uid` that is not recorded is excluded from the comparison, and its count is shown. A difference in chunks alone does not let you conclude that theme coverage or accuracy improved.

Source: [Scope of Topic Graph verification](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#review-settings-and-verification-limits) / [activity and references in retrieval](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve)

In [ ]:
if topic_results:
    passages_by_mode = {kind: topic_passages(topic_results[kind]) for kind in TOPIC_MODE_LABELS if kind in topic_results}
    uid_sets = {kind: {row["uid"] for row in passages if row["uid"] is not None}
                for kind, passages in passages_by_mode.items()}
    baseline = uid_sets.get("none")
    table([{"Method": kind, "Fragments returned": len(passages_by_mode[kind]),
            "Source documents (retrieved)": len({row["Source URL"] for row in passages_by_mode[kind] if row["Source URL"]}),
            "Confirmed unique chunks": len(uids), "Shared with none": len(uids & baseline) if baseline is not None else "Not tallied",
            "Chunks not in none": len(uids - baseline) if baseline is not None else "Not tallied",
            "uid not recorded": sum(row["uid"] is None for row in passages_by_mode[kind])}
           for kind, uids in uid_sets.items()])
    display_topic_rank_table(passages_by_mode)
    if baseline is None:
        print("The chunk differences are shown after the baseline none has been searched.")
    elif len(passages_by_mode) == 1:
        print("The differences from the baseline none are shown after balanced or diverse has been searched.")
    else:
        differences = [{"Method": kind, "Reference ID": row["Reference ID"], "Document": row["Document"], "Excerpt": topic_excerpt(row["Source text"])}
                       for kind, passages in passages_by_mode.items() if kind != "none" for row in passages
                       if row["uid"] is not None and row["uid"] not in baseline]
        if differences:
            print("Excerpts of fragments that are not in the baseline none:")
            table(differences[:5])
            if len(differences) > 5:
                print(f"Showing the first 5 of {len(differences)} differences. The full text remains in topic_results.")
        else:
            print("Within the confirmed chunk IDs, there are no fragments that are not in the baseline none.")
else:
    print("Not run: run the search cells (2.5-2.7) of the methods you want to compare first.")

### 2.8.1 Check the activity log of each method

For the methods that have been retrieved, the notebook shows a list of the steps and a chart of the elapsed time. The original `activity` JSON is kept collapsed.

This knowledge base connects only a `searchIndex` type knowledge source that uses the Topic Graph. It is normal that `searchIndex` is recorded in `activity[].type` and `graphSearch` does not appear. The absence of `graphSearch` does not mean that the Topic Graph is not used. Conversely, a `searchIndex` record alone does not let you confirm which topics were used to select candidates.

When you search relationships with an Entity Graph-enabled knowledge source, the knowledge base converts the question into GQL and runs it. Running GQL directly against the topic index with `docs/search.graph` is also a separate operation from the `retrieve` in this notebook.

Source: [Topic Graph retrieval configuration](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph) / [Activity log of the Retrieve API](https://learn.microsoft.com/rest/api/searchservice/knowledge-retrieval/retrieve?view=rest-searchservice-2026-11-01-preview) / [Querying an Entity Graph through a knowledge base](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-entity-graph#query-the-graph-through-a-knowledge-base) / [API for running GQL directly](https://learn.microsoft.com/rest/api/searchservice/documents/graph?view=rest-searchservice-2026-11-01-preview)

In [ ]:
import math
from datetime import datetime


def display_activity(activities: list[dict], title: str, *, show_raw: bool = True) -> None:
    if not isinstance(activities, list):
        raise TypeError("activity must be an array. Check the SDK response.")
    print(title, "all activity records:")
    if show_raw:
        show_json(activities)
    else:
        raw = html.escape(json.dumps(activities, ensure_ascii=False, indent=2))
        display(HTML(f'<details><summary>Open the original activity JSON</summary><pre>{raw}</pre></details>'))
    if not activities:
        print("There are no activity records, so the processing cannot be checked. Check include_activity.")
        return
    labels = {
        "modelQueryPlanning": "Search plan generation", "graphSearch": "GQL generation and graph search",
        "searchIndex": "Index search", "azureBlob": "Blob source search",
        "agenticReasoning": "Reasoning settings and usage", "modelAnswerSynthesis": "Answer generation",
        "modelWebSummarization": "Web search result summarization", "imageServing": "Image retrieval and delivery",
    }

    def timestamp(value: str | None) -> datetime | None:
        if value is None:
            return None
        parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
        if parsed.tzinfo is None:
            raise ValueError("The activity timestamp has no time zone.")
        return parsed

    rows, records = [], []
    for activity in activities:
        kind = activity.get("type") or "Not recorded"
        graph = activity.get("graphSearchArguments") or {}
        model = activity.get("model") or {}
        start, end = timestamp(activity.get("startedAt")), timestamp(activity.get("completedAt"))
        if start is not None and end is not None and end < start:
            raise ValueError(f"activity {activity.get('id')}: the end time is earlier than the start time.")
        elapsed = activity.get("elapsedMs")
        basis = "elapsedMs"
        if elapsed is None and start is not None and end is not None:
            elapsed = (end - start).total_seconds() * 1000
            basis = "calculated from start and end times"
        if elapsed is not None and (isinstance(elapsed, bool) or not isinstance(elapsed, (int, float)) or not math.isfinite(elapsed) or elapsed < 0):
            raise ValueError(f"activity {activity.get('id')}: elapsedMs is invalid.")

        def metric(key: str) -> object:
            value = activity.get(key)
            return graph.get(key) if value is None else value

        row = {
            "ID": activity.get("id"), "Type": kind, "Processing": labels.get(kind, "Unregistered processing type"),
            "Source": activity.get("knowledgeSourceName"), "Model": model.get("modelName") or graph.get("modelName"),
            "Time (ms)": elapsed, "Time basis": basis if elapsed is not None else None,
            "Model time (ms)": graph.get("modelElapsedMs"),
            "Input tokens": metric("inputTokens"), "Output tokens": metric("outputTokens"),
            "Reasoning tokens": activity.get("reasoningTokens"), "Count": activity.get("count"),
            "Attempts": graph.get("attempts"), "Outcome / error": activity.get("error") or graph.get("outcome"),
        }
        rows.append({key: "Not recorded" if value is None else value for key, value in row.items()})
        records.append((activity, start, end, elapsed))
    table(rows)
    bounds = [(start, end) for _, start, end, _ in records if start is not None and end is not None]
    origin = min(start for start, _ in bounds) if bounds else None
    span = (max(end for _, end in bounds) - origin).total_seconds() * 1000 if origin is not None else max((elapsed for _, _, _, elapsed in records if elapsed is not None), default=0)
    caption = "Timeline: elapsed ms since the first start time" if origin is not None else "Time comparison: actual start times, order and parallelism are not recorded"
    print(caption)
    left, plot_width, height = 330, 550, 95 + 44 * len(records)
    scale = plot_width / max(span, 1)
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1040 {height}" width="100%" role="img" aria-label="Processing time of {html.escape(title)}">',
             f'<text x="10" y="22" font-family="sans-serif" font-size="14">{html.escape(caption)}</text>']
    ticks = [span * index / 4 for index in range(5)] if span else [0]
    for tick in ticks:
        x = left + tick * scale
        parts.append(f'<line x1="{x}" y1="45" x2="{x}" y2="{height - 25}" stroke="#ddd"/>')
        parts.append(f'<text x="{x}" y="40" text-anchor="middle" font-family="sans-serif" font-size="12">{tick:.1f}</text>')
    for index, (activity, start, end, elapsed) in enumerate(records):
        y = 70 + 44 * index
        kind = activity.get("type") or "Not recorded"
        label = f"{activity.get('id', 'Not recorded')}: {labels.get(kind, kind)}"
        parts.append(f'<text x="10" y="{y + 5}" font-family="sans-serif" font-size="13">{html.escape(label)}</text>')
        if origin is not None and start is not None and end is not None:
            offset, duration = (start - origin).total_seconds() * 1000, (end - start).total_seconds() * 1000
        elif origin is None and elapsed is not None:
            offset, duration = 0, elapsed
        else:
            parts.append(f'<text x="{left}" y="{y + 5}" fill="#64748b" font-family="sans-serif" font-size="12">Start and end times not recorded</text>')
            continue
        x, width = left + offset * scale, duration * scale
        color = "#dc2626" if activity.get("error") or ((activity.get("graphSearchArguments") or {}).get("outcome") not in (None, "Success")) else "#3976b8"
        if duration == 0:
            parts.append(f'<circle cx="{x}" cy="{y}" r="4" fill="{color}"/>')
        else:
            parts.append(f'<rect x="{x}" y="{y - 9}" width="{width}" height="18" rx="3" fill="{color}"/>')
        parts.append(f'<text x="{left + plot_width + 12}" y="{y + 5}" font-family="sans-serif" font-size="12">{duration:.1f} ms</text>')
    display(SVG("".join(parts) + "</svg>"))

In [ ]:
if topic_results:
    for kind in TOPIC_MODE_LABELS:
        if kind in topic_results:
            display_activity(topic_results[kind]["activity"], f"Topic Graph: {kind}", show_raw=False)
else:
    print("Not run: run the search cells (2.5-2.7) of the methods whose activity log you want to check first.")

## 3. Reading the results and isolating problems

| Observed state | What to check next |
|---|---|
| `403` | Check separately the running user's Search permissions, the model permissions of the Search managed identity, and the network conditions. |
| `206` | A partial response. This notebook does not accept it as a success. Investigate from the displayed text and the request ID. |
| `412` | The settings may have been changed after the plan. Reread the definition from 2.1 and review a new plan. Do not switch to an unconditional PUT. |
| No source text | Check `includeReferenceSourceData` and the source's text fields. |
| The Topic Graph is not connected | Check the `topicGraphProfile` of the search-target field → the `indexName` of the profile, and the vectorizer. |
| The three Topic methods return the same results | With a small number of documents, there may be no difference. Do not interpret no difference as a failure, and do not interpret it as proof of the same quality. |

### Changes that persist and what to do after you finish

If you approved the first write in a live run, the dedicated `ft-topic-agentic-ks` and `ft-topic-agentic-kb`, and the search settings of `ft-topic-src`, remain. The topic documents are not re-registered. The settings changes remain even after you shut down the kernel.

Deletion and restoration of settings are not automated. When you no longer need them, check what this notebook newly created and its dependencies, and then clean up the dedicated KB first and the dedicated source second. Do not delete `ft-topic-src`, because Functional Test 3, Part B also uses it. When you restore settings, also check the latest definition and ETag so that you do not overwrite changes added later.

Notebook outputs can contain source text and the connection target. Clear the outputs before sharing or committing.

Source: [Troubleshooting Topic Graph](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-topic-graph#troubleshooting) / [Permissions for search](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve#prerequisites) / [Concurrency control](https://learn.microsoft.com/azure/search/search-howto-concurrency)

In [ ]:
if RUN_AZURE:
    for client in azure_clients:
        client.close()
    credential.close()
    print("Closed the SDK and credential clients. No Azure resources were deleted.")
else:
    print("No Azure credential client was created.")